# AgroMind: every method on the same data, and why we chose YieldRank

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ravixalgorithm/agromind/blob/main/notebooks/AgroMind_Method_Comparison.ipynb)

Every method below is trained and tested on **the same 43,443 district-season-year queries, the same 5 folds and the same calibration sets**
(DES APY 1997-98 to 2022-23 + NASA POWER + SoilGrids). The methods are:
* classic baselines (crop prior, last-year persistence, Gaussian NB, Random Forest, XGBoost, MLP);
* **reimplementations of six recent papers** with their published hyper-parameters: Dey 2024, Afzal 2025, Gunasekaran 2025, Ahmed 2025, Mahesh 2026 and Abu Bakr 2025 (fidelity note per method);
* ranking variants on the same (district-season, crop) pairs, and **YieldRank v4 (final)**;
* ablations of YieldRank.

Splits: random 5-fold, **unseen states** (leave-states-out, grouped by `state_group`) and **unseen years** (5 year blocks).
Hit@1 = the top-ranked crop is the one that yielded best. The full runs come from `experiments/compare.py` (about 2 CPU-hours) and are loaded here.
Section 5 re-trains one unseen-state fold live to show the saved numbers reproduce.

In [1]:
import os, sys, json, subprocess, urllib.request
from pathlib import Path
RAW = "https://raw.githubusercontent.com/ravixalgorithm/agromind/main/"
root = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "experiments/run.py").exists()), None)
if root is None:  # Colab
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "lightgbm", "xgboost", "catboost", "pyarrow"], check=True)
    root = Path("agromind")
    files = ["experiments/run.py", "experiments/compare.py", "experiments/crossconf.py", "backend/data/large/agromind_india_large.parquet",
             "docs/research/results.json"] + [f"experiments/results/compare_{p}_{t}.json" for p in ("random", "leave-states-out", "leave-years-out") for t in ("light", "heavyA", "heavyB")]
    for f in files:
        (root / f).parent.mkdir(parents=True, exist_ok=True)
        if not (root / f).exists():
            urllib.request.urlretrieve(RAW + f, root / f)
sys.path.insert(0, str(root / "experiments"))
import numpy as np, pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 80); pd.set_option("display.max_colwidth", 60)
SPLITS = ["random", "leave-states-out", "leave-years-out"]
F = pd.DataFrame([r | {"split": p} for p in SPLITS for t in ("light", "heavyA", "heavyB")
                  for r in json.load(open(root / f"experiments/results/compare_{p}_{t}.json"))
                  if (root / f"experiments/results/compare_{p}_{t}.json").exists()])
print(f"{F.method.nunique()} methods x {F.split.nunique()} splits x {F.fold.nunique()} folds = {len(F)} trained models")

26 methods x 3 splits x 5 folds = 390 trained models


## 1. Raw per-fold results (every number is one trained model on one test fold)

In [2]:
for p in SPLITS:
    print(f"\n==== {p} ====")
    for _, r in F[F.split == p].sort_values(["method", "fold"]).iterrows():
        print(f"fold {r.fold} | {r.method:44s} | Hit@1 {r.hit1:5.1f}% | Hit@3 {r.hit3:5.1f}% | NDCG@3 {r.ndcg3:.3f} | cold Hit@1 {r.hit1_cold:5.1f}% "
              f"| 90% set {r.conf_cov:5.1f}% / {r.conf_size:.2f} crops | train {r.train_s:6.1f}s")


==== random ====
fold 0 | CNN-LSTM (Bakr 2025)                         | Hit@1  27.4% | Hit@3  65.8% | NDCG@3 0.780 | cold Hit@1  25.1% | 90% set  90.3% / 4.69 crops | train   25.1s
fold 1 | CNN-LSTM (Bakr 2025)                         | Hit@1  24.1% | Hit@3  64.2% | NDCG@3 0.774 | cold Hit@1  21.2% | 90% set  89.4% / 4.60 crops | train   13.5s
fold 2 | CNN-LSTM (Bakr 2025)                         | Hit@1  25.3% | Hit@3  67.0% | NDCG@3 0.781 | cold Hit@1  22.4% | 90% set  89.8% / 4.62 crops | train   18.8s
fold 3 | CNN-LSTM (Bakr 2025)                         | Hit@1  26.0% | Hit@3  65.0% | NDCG@3 0.780 | cold Hit@1  22.0% | 90% set  90.0% / 4.64 crops | train   21.2s
fold 4 | CNN-LSTM (Bakr 2025)                         | Hit@1  25.5% | Hit@3  63.8% | NDCG@3 0.771 | cold Hit@1  21.6% | 90% set  89.5% / 4.62 crops | train   23.6s
fold 0 | CatBoost (Mahesh 2026 config)                | Hit@1  53.7% | Hit@3  85.0% | NDCG@3 0.902 | cold Hit@1  43.9% | 90% set  90.5% / 3.31 crops | train 

## 2. Summary table (mean over 5 folds), sorted by Hit@1 on unseen states

In [3]:
S = F.pivot_table(index="method", columns="split", values="hit1", aggfunc="mean")[SPLITS].round(1)
S.columns = [f"Hit@1 {c}" for c in S.columns]
lso = F[F.split == "leave-states-out"].groupby("method")
S["NDCG@3 states"] = lso.ndcg3.mean().round(3); S["cold Hit@1 states"] = lso.hit1_cold.mean().round(1)
S["90% cov/size states"] = (lso.conf_cov.mean().round(1).astype(str) + " / " + lso.conf_size.mean().round(2).astype(str))
S["train s"] = F.groupby("method").train_s.mean().round(1); S["infer ms/q"] = F.groupby("method").infer_ms.mean().round(3)
meta = F.drop_duplicates("method").set_index("method")
S["source"] = meta.source; S["family"] = meta.family
S.sort_values("Hit@1 leave-states-out", ascending=False)

,Hit@1 random,Hit@1 leave-states-out,Hit@1 leave-years-out,NDCG@3 states,cold Hit@1 states,90% cov/size states,train s,infer ms/q,source,family
method,,,,,,,,,,
YieldRank (ours),60.0,56.3,58.8,0.922,47.7,87.7 / 2.69,22.8,0.228,ours,pair ranker
YieldRank - persistence term,60.0,56.0,58.7,0.922,47.7,87.5 / 2.68,21.4,0.237,ours,ablation
LightGBM pointwise,59.4,55.7,58.3,0.921,47.1,88.4 / 2.69,9.4,0.128,ours,pair model
XGBoost LambdaMART,60.0,55.6,58.3,0.919,47.4,89.9 / 2.83,14.0,0.083,ours,pair ranker
LambdaMART (v4 listwise),59.7,55.4,57.9,0.920,47.4,89.8 / 2.87,9.5,0.174,ours,pair ranker
LightGBM XE-NDCG ranker,60.1,55.0,58.2,0.917,46.8,88.4 / 2.7,10.8,0.117,ours,pair ranker
Persistence,55.3,54.4,55.1,0.910,42.6,89.0 / 3.05,0.1,0.033,baseline,naive
MLP,53.7,38.3,50.3,0.830,32.0,75.1 / 3.07,5.4,0.017,baseline,classifier
CatBoost (Mahesh 2026 config),53.5,37.6,50.6,0.847,32.0,79.6 / 3.5,293.1,0.047,mahesh2026,paper


## 3. Is YieldRank significantly better? (paired Wilcoxon over test queries, all 5 folds)

In [4]:
R = json.load(open(root / "docs/research/results.json")).get("method_comparison")
if R:
    for m in R["methods"]:
        if m["p_vs_ours"]:
            print(f"{m['name']:44s} " + "  ".join(f"{s}: {m['hit1'][s]:5.1f}% (p={m['p_vs_ours'][s]:.1e})" for s in R["splits"]))
else:
    print("results.json has no method_comparison yet (run experiments/report.py)")

Crop prior                                   random:  29.0% (p=0.0e+00)  leave-states-out:  20.2% (p=0.0e+00)  leave-years-out:  27.4% (p=0.0e+00)
Persistence                                  random:  55.3% (p=5.6e-124)  leave-states-out:  54.4% (p=8.7e-20)  leave-years-out:  55.1% (p=8.7e-72)
Gaussian NB                                  random:  33.1% (p=0.0e+00)  leave-states-out:  27.8% (p=0.0e+00)  leave-years-out:  31.7% (p=0.0e+00)
Random Forest                                random:  57.5% (p=1.9e-30)  leave-states-out:  34.5% (p=0.0e+00)  leave-years-out:  52.0% (p=2.4e-176)
XGBoost                                      random:  57.5% (p=8.9e-32)  leave-states-out:  36.7% (p=0.0e+00)  leave-years-out:  52.0% (p=9.1e-180)
MLP                                          random:  53.7% (p=4.4e-173)  leave-states-out:  38.3% (p=0.0e+00)  leave-years-out:  50.3% (p=2.8e-280)
Stacking RF+ET+XGB (Gunasekaran 2025)        random:  53.3% (p=7.2e-174)  leave-states-out:  33.7% (p=0.0e+00)  l

## 4. Ablation: what each part of YieldRank buys on unseen states, and why we chose it

In [5]:
if R:
    for a in R["ablation"]:
        c = a["conformal"]
        print(f"- {a['removed']:30s} -> Hit@1 {a['hit1']['leave-states-out']:5.1f}%  NDCG@3 {a['ndcg3']['leave-states-out']:.3f}  "
              f"90% set {c['cov']}% / {c['size']} crops   ({a['note']})")
    print("\nWhy YieldRank:")
    for w in R["why"]:
        print(" *", w)

- nothing                        -> Hit@1  56.3%  NDCG@3 0.922  90% set 87.7% / 2.69 crops   (history + yield target + listwise/pointwise blend + persistence term + conformal sets)
- district history features      -> Hit@1  27.4%  NDCG@3 0.788  90% set 69.0% / 3.13 crops   (climate + soil + season only)
- yield target                   -> Hit@1  23.2%  NDCG@3 0.772  90% set 89.4% / 4.6 crops   (relevance = current area share (what is grown most) instead of yield percentile)
- persistence term               -> Hit@1  56.0%  NDCG@3 0.922  90% set 87.5% / 2.68 crops   (listwise + pointwise blend only)
- listwise ranker + persistence  -> Hit@1  55.7%  NDCG@3 0.921  90% set 88.4% / 2.69 crops   (pointwise regression on yield percentile alone)
- conformal step (top-1 only)    -> Hit@1  56.3%  NDCG@3 0.922  90% set 56.3% / 1.0 crops   (a single recommendation is right only Hit@1 of the time and carries no coverage guarantee)
- conformal step (fixed top-3)   -> Hit@1  56.3%  NDCG@3 0.922  90% 

## 5. Live check: retrain unseen-states fold 0 for three methods and compare with the saved numbers

In [6]:
import run, compare
df = run.load(); crops = sorted(df.crop_slug.unique()); QX, Qy, Q = run.query_table(df, crops, True)
from sklearn.model_selection import GroupShuffleSplit
rng = np.random.default_rng(run.SEED); idx = lambda qs: df.index[df.qid.isin(qs)]
for protocol in SPLITS:  # replay run.py's random-number use so the calibration sets match exactly
    for f, (trq, teq, groups) in enumerate(run.folds(df, protocol)):
        a, b = next(GroupShuffleSplit(1, test_size=0.2, random_state=run.SEED + f).split(trq, groups=groups.loc[trq].values))
        iid = rng.choice(trq[a], size=len(trq[b]), replace=False)
        if protocol == "leave-states-out" and f == 0:
            fit_i, te_i = idx(np.setdiff1d(trq[a], iid)), idx(teq)
for name in ["Persistence", "XGBoost (Dey 2024)", "YieldRank (ours)"]:
    score = compare.METHODS[name][-1](df, fit_i, crops, QX, Qy)
    live = 100 * run.per_query(df.loc[te_i], score(te_i))["hit@1"].mean()
    saved = F[(F.split == "leave-states-out") & (F.fold == 0) & (F.method == name)].hit1.iloc[0]
    print(f"{name:24s} live Hit@1 {live:5.1f}%   saved {saved:5.1f}%   {'OK' if abs(live - saved) < 0.15 else 'DIFF'}")

Persistence              live Hit@1  50.5%   saved  50.5%   OK


XGBoost (Dey 2024)       live Hit@1  33.7%   saved  33.7%   OK


YieldRank (ours)         live Hit@1  52.4%   saved  52.4%   OK
